[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/3-5_%EC%84%A0%ED%83%9D%EC%8B%AC%ED%99%94_%EC%8B%9C%EA%B3%84%EC%97%B4%EA%B8%B0%EC%B4%88%EB%AA%A8%EB%8D%B8.ipynb)

# 3-5 · [선택 심화] 시계열 기초모델(PatchTST)로 미래 전류를 예측해 RUL 구하기

**⏱ 40~60분 · 선택** — 3-4까지 마친 학생, 또는 교수자 시연용입니다. 본 과정의 필수 내용은 아닙니다.

## 🎯 이번 시간의 질문

> ChatGPT처럼 **엄청난 양의 데이터로 미리 학습한 AI 모델**을 우리 필터에 **그대로** 쓰면 RUL을 더 잘 맞힐까요?

최근에는 수많은 분야의 시계열(전력, 교통, 날씨, 센서 …)로 미리 학습한 **시계열 기초모델(Time-Series Foundation Model)**이 공개되고 있습니다.
우리 데이터로 따로 학습하지 않고(**zero-shot**) 과거 값만 넣으면 미래 값을 예측합니다. 여기서는 IBM이 2026년 공개한 **PatchTST-FM**을 씁니다.

| | 3-2 추세 외삽 | 3-3 랜덤 포레스트 | 3-5 기초모델 |
|---|---|---|---|
| 무엇을 예측? | 미래 HI (직선) | **RUL을 직접** | **미래 전류 값** (범위 포함) |
| 우리 데이터로 학습? | 기준선만 | 33개 필터로 학습 | **학습 안 함** (zero-shot) |
| RUL 계산 | 직선이 기준선에 닿는 시각 | 모델 출력 | 예측 경로가 기준선에 닿는 시각 |

기초모델은 미래 전류를 한 줄이 아니라 **분위수**로 줍니다. 예) 50% 분위수 = 가운데 예측, 90% 분위수 = "높게 나올 수 있는 쪽"(비관적 시나리오).

## 0. 모델 설치 (2~5분)

아래 셀은 모델 실행 도구(`granite-tsfm`)를 설치합니다. **설치나 다운로드가 실패해도 괜찮습니다.** 같은 모델로 미리 계산해 둔 예측 결과를 자동으로 불러와 실습을 이어 갑니다.
설치 후 "세션을 다시 시작하라"는 안내가 나오면 **런타임 → 세션 다시 시작**을 누르고, 이 셀은 건너뛰고 아래 준비 셀부터 실행하세요.

In [ ]:
# ▶ 그대로 실행하세요
%pip install -q granite-tsfm==0.3.9

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/filter_runs.csv",
             "data/filter_windows.csv",
             "data/filter_patchtst_forecasts.npz",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

In [ ]:
# ▶ 그대로 실행하세요
windows = pd.read_csv('data/filter_windows.csv')
main = windows[windows['status'] == 'main'].copy()
main['hi'] = main.groupby('run_id')['current_a'].transform(lambda s: s.rolling(3, min_periods=1).mean())
test = main[main['split'] == 'test']
TH = 1.07
saved = np.load('data/filter_patchtst_forecasts.npz')

# 기초모델 불러오기 (처음 한 번 약 1.5GB 다운로드). 실패하면 저장된 예측 결과를 사용.
try:
    import logging, torch
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()     # 설치·로딩 안내 메시지 줄이기
    logging.getLogger('tsfm_public').setLevel(logging.WARNING)
    from tsfm_public import PatchTSTFMForPrediction
    logging.getLogger('tsfm_public.models.patchtst_fm.modeling_patchtst_fm').setLevel(logging.WARNING)
    model = PatchTSTFMForPrediction.from_pretrained('ibm-granite/granite-timeseries-patchtst-fm-r2',
                                                    revision='b125275f9204d37cbb81fe47b9cf5e08a521e829').eval()
    MODE = '실제 모델 실행'
except Exception as e:
    model = None
    MODE = f'저장된 예측 결과 사용 ({type(e).__name__})'

def forecast(run, t):
    """t초까지의 전류로 앞으로 120칸(1,200초)을 예측. 결과: (120, 3) = 미래 시점 × [10%, 50%, 90%] 분위수."""
    if model is not None:
        history = run.loc[run['time_s'] <= t, 'current_a'].to_numpy()
        x = torch.tensor(history[None, :, None], dtype=torch.float32)
        with torch.inference_mode():
            out = model(past_values=x, prediction_length=120)
        q = np.sort(out.quantile_outputs[0, :, :, 0].cpu().numpy().T, axis=1)    # (120, 99)
        return q[:, [9, 49, 89]]
    return saved[f'r{run["run_id"].iloc[0]}_t{t}']

print('실행 방식:', MODE)

## 1. 한 필터의 미래 전류 예측하기

평가용 **17번 필터**(F = 473초)가 지금 **300초**째라고 합시다. 300초까지의 전류를 넣고 앞으로 1,200초를 예측합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
run = test[test['run_id'] == 17]
t_now = 300
q = forecast(run, t_now)
future_t = t_now + np.arange(1, 121) * 10
print(q.shape)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


**P4 · 예측 범위 그리기** — `plt.fill_between(x, 아래, 위)`는 두 선 사이를 색칠합니다. 10%~90% 분위수 사이를 칠해 **예측 범위**를 표시합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
past = run[run['time_s'] <= t_now]
plt.plot(run['time_s'], run['current_a'], color='lightgray', label='실제 전류 (정답)')
plt.plot(past['time_s'], past['current_a'], color='black', label='모델에 넣은 과거')
plt.plot(future_t, q[:, 1], color='C0', label='예측 50%')
plt.fill_between(future_t, q[:, 0], q[:, 2], alpha=0.3, label='예측 10~90%')
plt.axhline(TH, color='red', label='고장 기준선')
plt.axvline(473, color='black', linestyle=':', label='실제 고장')
plt.xlim(0, 900)
plt.xlabel('경과 시간 (초)')
plt.ylabel('전류 (A)')
plt.legend(fontsize=8)
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 전류가 0.6 → 0.9 A로 올라오는 중인데, 기초모델의 가운데 예측(파랑)은 잠시 오르다가 **다시 0.6~0.7 A로 내려간다**고 봅니다. 실제로는 계속 올라 473초에 고장 났습니다.
기초모델은 여러 분야의 시계열에서 "튀어 오른 값은 평소 수준으로 **돌아오는**" 패턴을 많이 배웠습니다. 하지만 **필터 막힘은 되돌아가지 않습니다.** 모델은 이 물리 지식을 모릅니다.

## 2. 예측 경로에서 RUL 구하기

예측 경로가 기준선(1.07 A)에 **처음 닿는 칸**을 찾아 RUL로 바꿉니다. 1,200초 안에 닿지 않으면 "1,200초 이상"으로 둡니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
def rul_from_path(path, now_hi, TH=1.07):
    if now_hi >= TH:                      # 이미 기준선을 넘었으면 0
        return 0.0
    hit = np.where(path >= TH)[0]         # 기준선 이상인 칸의 번호들
    return float((hit[0] + 1) * 10) if len(hit) else 1200.0

now_hi = past['hi'].iloc[-1]
print('50% 경로 RUL:', rul_from_path(q[:, 1], now_hi), '초')
print('90% 경로 RUL:', rul_from_path(q[:, 2], now_hi), '초')
print('실제 RUL:', 473 - t_now, '초')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 가운데(50%)와 비관적(90%) 경로 모두 1,200초 안에 기준선에 닿지 않아 "1,200초 이상"이 나왔습니다. 실제 RUL은 173초입니다.

## 3. 평가용 8개 필터 전체 평가

60초마다(60, 120, … 초) 예측해 RUL 오차를 구간별로 봅니다. (▶ 실제 모델 실행이면 1~2분)

In [ ]:
# ▶ 그대로 실행하세요
rows = []
for rid, r in test.groupby('run_id'):
    F = r['failure_s'].iloc[0]
    for t in range(60, F, 60):
        qq = forecast(r, t)
        now = r.loc[r['time_s'] <= t, 'hi'].iloc[-1]
        rows.append({'run_id': rid, 't': t, 'true': F - t,
                     'rul50': rul_from_path(qq[:, 1], now), 'rul90': rul_from_path(qq[:, 2], now)})
res = pd.DataFrame(rows)
res['구간'] = pd.cut(res['true'], [0, 100, 200, 300, 5000], right=False,
                    labels=['0~100초', '100~200초', '200~300초', '300초 이상'])
pd.DataFrame({'50% 경로': (res['rul50'] - res['true']).abs().groupby(res['구간']).mean().round(),
              '90% 경로(비관)': (res['rul90'] - res['true']).abs().groupby(res['구간']).mean().round(),
              '시점 수': res.groupby('구간').size()})

✅ **체크포인트** — 50% 경로 401 / 695 / 676 / 517초, 90% 경로 210 / 496 / 501 / 535초.

💬 **결과 해설** — 3-3의 랜덤 포레스트(0~300초 구간 62~84초)는 물론 3-2의 추세 외삽보다도 **훨씬 크게 틀립니다.** 대부분 "1,200초 안에 기준선에 안 닿는다"고 예측했기 때문입니다.
- **최신·대형 모델이라도 우리 설비 데이터에서 검증하기 전에는 믿을 수 없습니다.** 이번 경우는 "열화는 되돌아가지 않는다"는 도메인 특성을 모델이 모릅니다.
- 그래도 **90% 경로(비관적 시나리오)**는 고장 직전 구간에서 50% 경로보다 훨씬 낫습니다. 분위수를 주는 모델은 "나쁜 쪽 시나리오"를 경보에 쓸 수 있습니다(과제 2).
- 현장에서는 이런 모델을 **우리 데이터로 추가 학습(fine-tuning)**하거나, 물리 제약(단조 증가)을 더해 쓰는 연구가 진행 중입니다.

---
## 4. 과제 (15분)

### 📝 과제 1 · 다른 필터·시점 보기
22번 필터(F = 799초)의 **420초**와 **660초** 시점 예측을 1절처럼 그려 보세요. 모델이 오르는 추세를 따라가는 경우도 있나요?

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: run = test[test['run_id'] == 22]; t_now = 420; q = forecast(run, t_now) 후 1절 그림 코드 (xlim, 고장선 799로)


### 📝 과제 2 · 비관적 시나리오로 경보 만들기
"**90% 경로의 RUL이 60초 이하**가 되면 교체"라는 규칙을 평가용 8개 필터에 적용하세요. 필터마다 첫 경보 시점과, 그때 실제로 남은 시간을 표로 만들고, 3-4처럼 **30초 이상 남기고 경보한** 필터가 몇 개인지 세어 보세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: alarm = res[res['rul90'] <= 60].groupby('run_id')['t'].min()  → 실제 남은 시간 = F − 경보 시각


### 📝 과제 3 · 생각해 보기
기초모델을 현장 CBM에 도입하자는 제안이 왔습니다. 오늘 결과를 근거로 **찬성 이유 1개, 우려 1개, 도입 전에 꼭 할 일 1개**를 적으세요.

✏️ **나의 답:**

---
## ✅ 정리

- 시계열 기초모델은 **학습 없이** 미래 센서값과 그 범위를 예측한다. RUL은 예측 경로가 기준선에 닿는 시각으로 **변환**해야 한다.
- 이번 필터 데이터에서는 "값이 평소 수준으로 돌아온다"고 예측해 RUL을 크게 틀렸다. **최신 모델도 우리 데이터로 검증하기 전에는 믿지 않는다.**
- 분위수 예측의 **비관적 시나리오**는 경보 규칙으로 활용할 수 있다.